## 1. Сгенерировать набор данных при помощи make_classification из sklearn.

In [1]:
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=2000, n_features=10, n_informative=6, n_redundant=2,
                           n_classes=2, random_state=42)

## 2. Построить нейронную сеть для решения задачи классификации.


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split

# Переводим данные в тензоры
X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y).float().unsqueeze(1)

# Формируем датасеты и делим на train/val
full_dataset = TensorDataset(X_tensor, y_tensor)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

# MLP-модель
class Net(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)  # Бинарный выход (логит)
        )
    def forward(self, x):
        return self.features(x)


## 3. Исследовать влияние различных классификаторов на качество модели (SGD, Adam, RMSProp), записать выводы.


In [3]:
from torch.optim import SGD, Adam, RMSprop
from sklearn.metrics import accuracy_score

def train_eval(opt_type, dropout=0.0):
    model = Net(dropout=dropout)
    if opt_type == 'sgd':
        optimizer = SGD(model.parameters(), lr=0.01)
    elif opt_type == 'adam':
        optimizer = Adam(model.parameters(), lr=0.01)
    elif opt_type == 'rmsprop':
        optimizer = RMSprop(model.parameters(), lr=0.01)
    criterion = nn.BCEWithLogitsLoss()
    # Обучение
    for epoch in range(20):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
    # Оценка на вал
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb)
            all_logits.append(logits)
            all_labels.append(yb)
    preds = torch.sigmoid(torch.cat(all_logits)).cpu().numpy() > 0.5
    y_true = torch.cat(all_labels).cpu().numpy()
    acc = accuracy_score(y_true, preds)
    return acc

acc_sgd = train_eval('sgd')
acc_adam = train_eval('adam')
acc_rmsprop = train_eval('rmsprop')
print(f'SGD: {acc_sgd:.3f}, Adam: {acc_adam:.3f}, RMSProp: {acc_rmsprop:.3f}')


SGD: 0.890, Adam: 0.917, RMSProp: 0.920


* Adam часто converge быстрее и достигает более высокой точности на небольших нейронных сетях.

* SGD обучается стабильнее при правильном подборе lr, но требует больше эпох для выхода на плато.

* RMSProp — часто лучше, чем SGD, но Adam часто лидирует на практических задачах. Итоговые значения точности нужно сравнить непосредственно после нескольких запусков.

## 4. Добавить к модели метод регуляризации Dropout и повторить пункт 3 уже с ним.


In [4]:
acc_sgd_dropout = train_eval('sgd', dropout=0.3)
acc_adam_dropout = train_eval('adam', dropout=0.3)
acc_rmsprop_dropout = train_eval('rmsprop', dropout=0.3)
print(f'SGD+Dropout: {acc_sgd_dropout:.3f}, Adam+Dropout: {acc_adam_dropout:.3f}, RMSProp+Dropout: {acc_rmsprop_dropout:.3f}')

SGD+Dropout: 0.882, Adam+Dropout: 0.927, RMSProp+Dropout: 0.927


* Dropout обычно слегка снижает точность на train, но метрика на валидации улучшается либо стабилизируется.

* Особенно заметен выигрыш на небольших датасетах или если модель начинает переобучаться.